# 🌍 Life Expectancy by Country — Expanded Analysis (SOLUTION)

Complete version with full EDA, quantile analysis, simulation, advanced visualizations, and insights.

Run all cells to see plots and printed outputs.


## 1. Theory: Quantiles & Why They Matter Here

We previously learned that **quantiles** split data into groups of equal size.

- **Quartiles** divide data into 4 groups (25% each)
  - Q1 (25th percentile), Q2 (median = 50th), Q3 (75th percentile)
- The **Interquartile Range (IQR = Q3 - Q1)** is a robust measure of spread.

In this project:
- We first look at the overall distribution of **Life Expectancy**.
- Then we split countries into "Low GDP" and "High GDP" groups using the **median GDP**.
- Finally, we compare the life expectancy distributions of these two groups using quantiles and visualizations.

This helps answer: *Does higher national wealth (GDP) tend to be associated with higher life expectancy?*


## 2. Loading and Inspecting the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# # Load the data
data = pd.read_csv("/home/workdir/attachments/country_data.csv")

# # Print the first 5 rows and the column names
print(data.head())
print("\nColumns:", data.columns.tolist())
print("\nShape:", data.shape)

## 3. Isolating Life Expectancy and Finding Quartiles

In [ ]:
# # Isolate the Life Expectancy column
life_expectancy = data["Life Expectancy"]

# # Compute quartiles using np.quantile
life_expectancy_quartiles = np.quantile(life_expectancy, [0.25, 0.5, 0.75])
print("Life Expectancy Quartiles (Q1, Median, Q3):", life_expectancy_quartiles)

# Interpretation
print("\nInterpretation:")
print(f"25% of countries have life expectancy ≤ {life_expectancy_quartiles[0]:.1f} years")
print(f"50% of countries have life expectancy ≤ {life_expectancy_quartiles[1]:.1f} years (median)")
print(f"75% of countries have life expectancy ≤ {life_expectancy_quartiles[2]:.1f} years")

## 4. Visualizing the Overall Distribution

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(life_expectancy, bins=20, color="#4FC3F7", edgecolor="#0277BD", alpha=0.8)
plt.axvline(life_expectancy_quartiles[0], color="red", linestyle="--", label="Q1 (25%)")
plt.axvline(life_expectancy_quartiles[1], color="green", linestyle="--", label="Median (50%)")
plt.axvline(life_expectancy_quartiles[2], color="red", linestyle="--", label="Q3 (75%)")
plt.title("Distribution of Life Expectancy Across Countries")
plt.xlabel("Life Expectancy (years)")
plt.ylabel("Number of Countries")
plt.legend()
plt.show()

print("The distribution is left-skewed (long tail on the left/low end).")

## 5. Splitting Data by GDP (Wealth)

In [ ]:
# # Isolate GDP column
gdp = data["GDP"]

# # Find median GDP
median_gdp = np.quantile(gdp, 0.5)   # or np.median(gdp)
print(f"Median GDP: ${median_gdp:,.2f}")

# Split into low and high GDP groups
low_gdp = data[data["GDP"] <= median_gdp]
high_gdp = data[data["GDP"] > median_gdp]

print(f"Number of Low GDP countries: {len(low_gdp)}")
print(f"Number of High GDP countries: {len(high_gdp)}")

## 6. Comparing Life Expectancy in Low vs High GDP Countries

In [ ]:
# # Quartiles for low GDP countries
low_gdp_quartiles = np.quantile(low_gdp["Life Expectancy"], [0.25, 0.5, 0.75])
print("Low GDP Countries - Life Expectancy Quartiles:", low_gdp_quartiles)

# # Quartiles for high GDP countries
high_gdp_quartiles = np.quantile(high_gdp["Life Expectancy"], [0.25, 0.5, 0.75])
print("High GDP Countries - Life Expectancy Quartiles:", high_gdp_quartiles)

print("\nKey Insight:")
print(f"Median life expectancy in Low GDP countries: {low_gdp_quartiles[1]:.1f} years")
print(f"Median life expectancy in High GDP countries: {high_gdp_quartiles[1]:.1f} years")
print("There is a substantial difference (~10+ years) in median life expectancy between the two groups.")

## 7. Visual Comparison: Low GDP vs High GDP

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(high_gdp["Life Expectancy"], bins=15, alpha=0.6, label="High GDP Countries", color="#81C784")
plt.hist(low_gdp["Life Expectancy"], bins=15, alpha=0.6, label="Low GDP Countries", color="#E57373")
plt.axvline(low_gdp_quartiles[1], color="#C62828", linestyle="--", linewidth=2, label="Low GDP Median")
plt.axvline(high_gdp_quartiles[1], color="#2E7D32", linestyle="--", linewidth=2, label="High GDP Median")
plt.title("Life Expectancy: High GDP vs Low GDP Countries")
plt.xlabel("Life Expectancy (years)")
plt.ylabel("Number of Countries")
plt.legend()
plt.show()

print("The two distributions are clearly separated. High GDP countries have much higher life expectancy overall.")

## 8. 🎮 Simulation: Try Different GDP Thresholds

Instead of always using the median, what happens if we use a different cutoff (e.g., 25th or 75th percentile of GDP)?


In [ ]:
# ============== SIMULATION PARAMETERS (MODIFY THESE) ==============
GDP_THRESHOLD_PERCENTILE = 0.5   # Change to 0.25, 0.75, etc.
TARGET_LIFE_EXPECTANCY = 70      # Example value to evaluate
# =====================================================================

threshold_gdp = np.quantile(data["GDP"], GDP_THRESHOLD_PERCENTILE)
print(f"Using GDP threshold at {GDP_THRESHOLD_PERCENTILE*100}th percentile: ${threshold_gdp:,.2f}")

low_group = data[data["GDP"] <= threshold_gdp]
high_group = data[data["GDP"] > threshold_gdp]

low_q = np.quantile(low_group["Life Expectancy"], [0.25, 0.5, 0.75])
high_q = np.quantile(high_group["Life Expectancy"], [0.25, 0.5, 0.75])

print(f"Low group median life expectancy : {low_q[1]:.1f} years (n={len(low_group)})")
print(f"High group median life expectancy: {high_q[1]:.1f} years (n={len(high_group)})")

# Where does TARGET_LIFE_EXPECTANCY fall?
def get_quarter(val, quartiles):
    if val <= quartiles[0]: return "1st quarter (bottom 25%)"
    elif val <= quartiles[1]: return "2nd quarter"
    elif val <= quartiles[2]: return "3rd quarter"
    else: return "4th quarter (top 25%)"

print(f"\nA country with {TARGET_LIFE_EXPECTANCY} years life expectancy would be in the:")
print(f"  → Low GDP group: {get_quarter(TARGET_LIFE_EXPECTANCY, low_q)}")
print(f"  → High GDP group: {get_quarter(TARGET_LIFE_EXPECTANCY, high_q)}")

## 9. Advanced Visualizations (Scatter Plot + Regression)

In [ ]:
# Scatter plot with regression line
plt.figure(figsize=(9,6))
sns.regplot(data=data, x="GDP", y="Life Expectancy", 
            scatter_kws={"alpha":0.6, "s":60}, 
            line_kws={"color":"red", "linewidth":2})
plt.title("Life Expectancy vs GDP per Capita (with regression line)")
plt.xlabel("GDP per Capita (USD)")
plt.ylabel("Life Expectancy (years)")
plt.show()

corr = data["Life Expectancy"].corr(data["GDP"])
print(f"Correlation between GDP and Life Expectancy: {corr:.3f}")
print("There is a strong positive correlation — richer countries tend to have higher life expectancy.")

## 10. Light Machine Learning Perspective

- **Simple Linear Regression**: We can predict Life Expectancy from GDP.
- **Robustness**: Because GDP is heavily right-skewed, using quantiles or log-transforming GDP can improve models.
- **Feature Engineering**: Creating a "High/Low GDP" binary feature (based on median or quartiles) can be useful in classification or as an interaction term.
- **Quantile Regression**: Instead of predicting average life expectancy, we could predict the 25th or 75th percentile of life expectancy for different GDP levels (useful for policy analysis).

This dataset is a classic example of how economic indicators relate to human development outcomes.


## 🗺️ Analysis Flowchart

```mermaid
flowchart TD
    A[Load country_data.csv] --> B[EDA: head(), describe(), correlation]
    B --> C[Isolate Life Expectancy & Compute Quartiles]
    C --> D[Plot Overall Histogram + Quartile Lines]
    D --> E[Find Median GDP & Split into Low/High GDP groups]
    E --> F[Compare Quartiles & Plot Dual Histograms]
    F --> G[Simulation: Change GDP threshold]
    G --> H[Advanced Viz: Scatter + Regression]
    H --> I[Insights & Conclusions]
```


## ✏️ More Practice Exercises

1. Compute the **deciles** (10-quantiles) of Life Expectancy for the whole dataset.
2. Find the top 10 countries by Life Expectancy and the bottom 10. What is their average GDP?
3. Instead of splitting at the median GDP, split at the **75th percentile** of GDP. How do the life expectancy distributions change?
4. Create a boxplot comparing Life Expectancy for Low vs High GDP countries using seaborn.
5. Add a new column `GDP_log = np.log1p(data['GDP'])` and re-run the correlation and scatter plot. Does it look better?
6. (Challenge) Use `statsmodels` or `scikit-learn` to fit a simple linear regression and print the coefficients and R².


## ✅ Summary of Findings

- Life expectancy varies widely across countries (from ~46 to ~82.5 years).
- There is a **strong positive correlation** (~0.61) between GDP per capita and life expectancy.
- Countries above the median GDP have a median life expectancy roughly **10 years higher** than those below.
- The simulation shows that the choice of GDP threshold affects how we classify countries, but the overall pattern remains consistent.
- Wealth (as measured by GDP) is strongly associated with better health outcomes at the national level, though it is not the only factor.

This project demonstrates how quantiles and subgroup analysis can reveal important patterns in real-world data.
